# Retrieval-Augmented Generation (RAG) — End-to-End Demo

This notebook builds a complete, minimal RAG pipeline that runs **entirely locally**, except for the final answer generation, which calls **Groq's `llama-3.3-70b-versatile`**.

Pipeline:
1. **Load** a list of files from `data/` (`.txt`, `.md`, `.pdf`)
2. **Split** each file into overlapping chunks
3. **Embed** each chunk with a local sentence-transformer model
4. **Store** the embeddings in a local, native vector database ([ChromaDB](https://www.trychroma.com/), persisted to disk — no server required)
5. **Visualize** the vector space in 3D to see how chunks cluster by topic
6. **Ask a question** → embed it, retrieve the most relevant chunks
7. **Compile an answer** by sending the question + retrieved context to Groq's Llama 3.3 70B model

Only Groq is used as an external API (for generation) — everything else (chunking, embeddings, vector DB, visualization) runs locally.

## 0. Setup

This notebook expects to run under the `.venv` virtual environment in this project folder — select **Python (.venv)** as the kernel (top-right in VS Code / Jupyter) before running the cells below.

Dependencies are listed in `requirements.txt`; the cell below installs them into `.venv`. API keys are loaded from a local `.env` file via `python-dotenv` — copy `.env.example` to `.env` and fill in your Groq key (free tier at https://console.groq.com/keys):

```
GROQ_API_KEY=your-key-here
```

`.env` is never read by anything outside this notebook/process, so it keeps the key out of the notebook source itself.

In [ ]:
%pip install -q -r requirements.txt


In [ ]:
import os
import glob
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from sklearn.decomposition import PCA

import chromadb
from sentence_transformers import SentenceTransformer
from groq import Groq
from pypdf import PdfReader

load_dotenv()

DATA_DIR = Path("data")
CHUNK_SIZE = 500        # characters per chunk
CHUNK_OVERLAP = 80      # overlap between consecutive chunks
EMBED_MODEL_NAME = "all-MiniLM-L6-v2"
GROQ_MODEL = "llama-3.3-70b-versatile"
TOP_K = 4               # number of chunks to retrieve per question


## 1. Load documents

Drop `.txt`, `.md`, or `.pdf` files into the `data/` folder, then run the cell below to load them.

In [ ]:
DATA_DIR.mkdir(exist_ok=True)

def load_pdf(path):
    reader = PdfReader(path)
    return "\n".join(page.extract_text() or "" for page in reader.pages)

text_paths = sorted(glob.glob(str(DATA_DIR / "*.txt"))) + sorted(glob.glob(str(DATA_DIR / "*.md")))
pdf_paths = sorted(glob.glob(str(DATA_DIR / "*.pdf")))

documents = {}
for path in text_paths:
    documents[Path(path).name] = Path(path).read_text(encoding="utf-8", errors="ignore")
for path in pdf_paths:
    documents[Path(path).name] = load_pdf(path)

assert documents, f"No .txt/.md/.pdf files found in {DATA_DIR}/ — add some documents and re-run this cell."
print(f"Loaded {len(documents)} files: {list(documents.keys())}")


## 2. Split into chunks

A simple fixed-size sliding-window splitter with overlap. Each chunk keeps track of its source file so answers can be traced back to a document.

In [ ]:
def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    text = " ".join(text.split())  # normalize whitespace
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

records = []  # each: {id, text, source}
for source, text in documents.items():
    for i, chunk in enumerate(chunk_text(text)):
        records.append({"id": f"{source}::{i}", "text": chunk, "source": source})

print(f"Created {len(records)} chunks from {len(documents)} documents")
pd.DataFrame(records)[["id", "source"]]


## 3. Compute embeddings

Using a small local sentence-transformer model (`all-MiniLM-L6-v2`, 384 dimensions) — no API calls, runs on CPU.

In [ ]:
embedder = SentenceTransformer(EMBED_MODEL_NAME)
texts = [r["text"] for r in records]
embeddings = embedder.encode(texts, show_progress_bar=True, normalize_embeddings=True)

for r, e in zip(records, embeddings):
    r["embedding"] = e

print(f"Embedding matrix shape: {embeddings.shape}")


## 4. Store in a local, native vector database

[ChromaDB](https://docs.trychroma.com/) persists to a folder on disk (`./chroma_db`) — no external server, no cloud account, no network calls.

In [ ]:
chroma_client = chromadb.PersistentClient(path="./chroma_db")
collection = chroma_client.get_or_create_collection(
    name="rag_demo",
    metadata={"hnsw:space": "cosine"},
)

# Reset collection contents so re-running the notebook doesn't duplicate chunks
existing_ids = collection.get()["ids"]
if existing_ids:
    collection.delete(ids=existing_ids)

collection.add(
    ids=[r["id"] for r in records],
    embeddings=[r["embedding"].tolist() for r in records],
    documents=[r["text"] for r in records],
    metadatas=[{"source": r["source"]} for r in records],
)

print(f"Stored {collection.count()} chunks in the vector DB")


## 5. Visualize the vector space

Embeddings live in 384 dimensions — we project them to 3D with PCA to see how chunks cluster by topic/source document. Points close together are semantically similar.

**Click and drag with the mouse to rotate the plot in 3D; scroll to zoom, right-click drag to pan.**

In [ ]:
pca = PCA(n_components=3)
coords = pca.fit_transform(np.array([r["embedding"] for r in records]))

viz_df = pd.DataFrame({
    "x": coords[:, 0],
    "y": coords[:, 1],
    "z": coords[:, 2],
    "source": [r["source"] for r in records],
    "preview": [r["text"][:80] + "..." for r in records],
})

fig = px.scatter_3d(
    viz_df, x="x", y="y", z="z",
    color="source", hover_data=["preview"],
    title="Chunk embeddings projected to 3D (PCA) — drag to rotate",
)
fig.update_traces(marker=dict(size=5))
fig.update_layout(scene=dict(dragmode="orbit"), margin=dict(l=0, r=0, b=0, t=40))
fig.show()


## 6. Ask a question — retrieve relevant chunks

Embed the question with the same model, then run a similarity search against the vector DB.

In [ ]:
def retrieve(question, k=TOP_K):
    q_embedding = embedder.encode([question], normalize_embeddings=True)[0]
    results = collection.query(query_embeddings=[q_embedding.tolist()], n_results=k)
    hits = []
    for text, meta, dist in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
        hits.append({"text": text, "source": meta["source"], "distance": dist})
    return hits, q_embedding

question = "How does Python differ from other programming languages?"
hits, q_embedding = retrieve(question)

for h in hits:
    print(f"[{h['source']}] (distance={h['distance']:.3f})\n{h['text']}\n")


### Bonus: see where the question lands in vector space

(also draggable — rotate to see how the question point relates to the retrieved cluster)

In [ ]:
q_coords = pca.transform(q_embedding.reshape(1, -1))

fig2 = px.scatter_3d(viz_df, x="x", y="y", z="z", color="source", hover_data=["preview"])
fig2.add_scatter3d(
    x=q_coords[:, 0], y=q_coords[:, 1], z=q_coords[:, 2],
    mode="markers", marker=dict(size=10, color="black", symbol="diamond"),
    name="question",
)
fig2.update_layout(
    title=f'Question: "{question}"',
    scene=dict(dragmode="orbit"),
    margin=dict(l=0, r=0, b=0, t=40),
)
fig2.show()


## 7. Compile an answer with Groq (`llama-3.3-70b-versatile`)

The retrieved chunks are stitched into a context block and sent to the LLM along with the question.

In [ ]:
groq_api_key = os.getenv("GROQ_API_KEY")
assert groq_api_key, "GROQ_API_KEY not found — copy .env.example to .env and add your key."
groq_client = Groq(api_key=groq_api_key)

def compile_answer(question, hits):
    context = "\n\n".join(f"[Source: {h['source']}]\n{h['text']}" for h in hits)
    prompt = f"""Answer the question using only the context below. If the context doesn't contain the answer, say so.

Context:
{context}

Question: {question}
Answer:"""

    response = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {"role": "system", "content": "You are a helpful assistant that answers strictly from the provided context and cites the source file(s) you used."},
            {"role": "user", "content": prompt},
        ],
        temperature=0.2,
    )
    return response.choices[0].message.content

answer = compile_answer(question, hits)
print(answer)


## 8. End-to-end `ask()` helper

Combine retrieval + generation into a single function for interactive use.

In [ ]:
def ask(question, k=TOP_K, verbose=True):
    hits, _ = retrieve(question, k=k)
    answer = compile_answer(question, hits)
    if verbose:
        print("Retrieved chunks:")
        for h in hits:
            print(f"  - [{h['source']}] {h['text'][:100]}...")
        print("\nAnswer:\n" + answer)
    return answer

# Try it yourself:
ask("What causes climate change?")


## Next steps

- Drop more files into `data/` (`.txt` / `.md` / `.pdf`) and re-run from Step 1
- Swap `all-MiniLM-L6-v2` for a larger embedding model for better retrieval accuracy
- Try UMAP instead of PCA for a visualization that better preserves cluster structure at scale
- Add a DOCX/HTML loader alongside the existing PDF/txt/md support for richer document sets
- Tune `CHUNK_SIZE`, `CHUNK_OVERLAP`, and `TOP_K` for your corpus